# Voting classifier
Hard and soft voting on the heart disease data, weights, and voting over one algorithm with different settings. The interactive decision-surface demo is `app.py` (`python app.py`, then open http://127.0.0.1:8050).

In [1]:
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.datasets import load_iris, make_classification
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

## 1. The heart disease data
303 patients, 13 features, target 1 = heart disease (the same file as Note 76).
Conditions: every score is stratified 10-fold cross-validation repeated 5 times with different shuffles;
logistic regression and KNN get standardized features.

In [2]:
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

heart = pd.read_csv("data/heart.csv", encoding="utf-8-sig")
X = heart.drop(columns="target")
y = heart["target"]
print(X.shape)
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=5, random_state=0)

(303, 13)


## 2. Each base model alone

In [3]:
clf1 = make_pipeline(StandardScaler(), LogisticRegression())
clf2 = RandomForestClassifier(random_state=42)
clf3 = make_pipeline(StandardScaler(), KNeighborsClassifier())

# estimators: a list of (name, model) pairs
estimators = [("lr", clf1), ("rf", clf2), ("knn", clf3)]

for name, model in estimators:
    scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
    print(name, np.round(np.mean(scores), 3))

lr 0.825


rf 0.817


knn 0.821


## 3. Hard and soft voting

In [4]:
vc = VotingClassifier(estimators=estimators, voting="hard")
print("hard", np.round(np.mean(cross_val_score(vc, X, y, cv=cv, scoring="accuracy")), 3))

vc_soft = VotingClassifier(estimators=estimators, voting="soft")
print("soft", np.round(np.mean(cross_val_score(vc_soft, X, y, cv=cv, scoring="accuracy")), 3))

hard 0.834


soft 0.841


## 4. Weights: give some models a bigger say
Try every weight from 1 to 3 for each model (27 combinations), with soft voting.

In [5]:
results = []
for i in range(1, 4):
    for j in range(1, 4):
        for k in range(1, 4):
            vc = VotingClassifier(estimators=estimators, voting="soft", weights=[i, j, k])
            acc = np.mean(cross_val_score(vc, X, y, cv=cv, scoring="accuracy"))
            results.append((i, j, k, round(acc, 3)))
weights = pd.DataFrame(results, columns=["lr", "rf", "knn", "accuracy"]).sort_values("accuracy", ascending=False)
print(weights[weights[["lr", "rf", "knn"]].eq(1).all(axis=1)])
weights.head(8)

   lr  rf  knn  accuracy
0   1   1    1     0.841


,lr,rf,knn,accuracy
25,3,3,2,0.844
10,2,1,2,0.843
11,2,1,3,0.842
16,2,3,2,0.842
19,3,1,2,0.842
23,3,2,3,0.842
22,3,2,2,0.842
13,2,2,2,0.841


## 4b. When one member is much stronger: iris, made harder
Keep only versicolor and virginica, and only the two sepal features, where the two species overlap a lot.
Logistic regression is far ahead of the other two, so the vote cannot beat it.

In [6]:
iris = load_iris(as_frame=True).frame.rename(columns={"target": "species"})
new_df = iris[iris["species"] != 0][["sepal length (cm)", "sepal width (cm)", "species"]]
Xi, yi = new_df.iloc[:, 0:2], new_df.iloc[:, -1]
iris_est = [("lr", LogisticRegression()), ("rf", RandomForestClassifier(random_state=42)), ("knn", KNeighborsClassifier())]
for name, model in iris_est + [("hard", VotingClassifier(iris_est, voting="hard")),
                               ("soft", VotingClassifier(iris_est, voting="soft")),
                               ("soft, weights 3,1,1", VotingClassifier(iris_est, voting="soft", weights=[3, 1, 1]))]:
    print(name, np.round(np.mean(cross_val_score(model, Xi, yi, cv=10)), 2))

lr 0.75


rf 0.6


knn 0.61


hard 0.67


soft 0.64


soft, weights 3,1,1 0.71


## 5. One algorithm, five settings
SVMs with a polynomial kernel of degree 1 to 5, on a synthetic dataset with 20 features.
Soft voting needs probabilities; in scikit-learn 1.9 an SVC gets them by wrapping it in CalibratedClassifierCV
(the older SVC(probability=True) is deprecated).

In [7]:
X, y = make_classification(n_samples=1000, n_features=20, n_informative=15, n_redundant=5, random_state=2)

svms = [(f"svm{d}", CalibratedClassifierCV(SVC(kernel="poly", degree=d), ensemble=False)) for d in range(1, 6)]
for name, model in svms:
    print(name, np.round(np.mean(cross_val_score(model, X, y, cv=10, scoring="accuracy")), 3))

svm1 0.851


svm2 0.855


svm3 0.894


svm4 0.854


svm5 0.871


In [8]:
vc = VotingClassifier(estimators=svms, voting="soft")
print("soft voting over the five SVMs", np.round(np.mean(cross_val_score(vc, X, y, cv=10, scoring="accuracy")), 3))

soft voting over the five SVMs 0.928


## 6. The decision-surface demo
The same code as `app.py`: three base models on the concentric-circles data, hard against soft voting.

In [9]:
from app import figure, run

for voting in ["hard", "soft"]:
    results, X_train, y_train = run("concentric circles", ["logistic regression", "Gaussian naive Bayes", "random forest"], voting)
    print(voting, [(t, round(a, 2)) for t, _, a in results])
figure(results, X_train, y_train)

hard [('voting (hard)', 0.66), ('logistic regression', 0.53), ('Gaussian naive Bayes', 0.6), ('random forest', 0.89)]


soft [('voting (soft)', 0.92), ('logistic regression', 0.53), ('Gaussian naive Bayes', 0.6), ('random forest', 0.89)]


In [10]:
# Why soft voting wins on the concentric circles: where hard and soft disagree, who is sure?
import numpy as np
from app import run, load
names = ["logistic regression", "Gaussian naive Bayes", "random forest"]
hard_vc = run("concentric circles", names, "hard")[0][0][1]
soft_vc = run("concentric circles", names, "soft")[0][0][1]
X_train, X_test, y_train, y_test = load("concentric circles")
d = hard_vc.predict(X_test) != soft_vc.predict(X_test)
print("disagree:", d.sum(), " soft right:", (soft_vc.predict(X_test)[d] == y_test[d]).sum())
rf = soft_vc.estimators_[2]
print("soft agrees with random forest:", (soft_vc.predict(X_test)[d] == rf.predict(X_test)[d]).mean())
for n, m in zip(names, soft_vc.estimators_):
    print(n, "mean |p - 0.5| on disagreements:", round(np.abs(m.predict_proba(X_test)[d, 1] - 0.5).mean(), 3))

disagree: 30  soft right: 28


soft agrees with random forest: 1.0
logistic regression mean |p - 0.5| on disagreements: 0.013
Gaussian naive Bayes mean |p - 0.5| on disagreements: 0.06
random forest mean |p - 0.5| on disagreements: 0.348
